## Deploy Genie Spaces

This notebook creates or updates Genie Spaces in the target workspace from exported configurations.

**What it does:**
1. Reads exported space configurations from the import volume
2. For each space: creates new or updates existing
3. Records source→target space ID mappings
4. Writes deployment manifest

**Prerequisites:**
- CAN USE on target SQL warehouse
- CAN MANAGE on target parent folder

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Setup Path Resolution

In [ ]:
import sys
import os

notebook_path = os.getcwd()
if notebook_path.startswith("/Workspace"):
    bundle_root = os.path.dirname(notebook_path)
else:
    bundle_root = os.path.dirname(os.path.abspath("__file__"))

helpers_path = os.path.join(bundle_root, "helpers")
if not os.path.exists(helpers_path):
    helpers_path = os.path.join(os.path.dirname(bundle_root), "helpers")

if helpers_path not in sys.path:
    sys.path.insert(0, os.path.dirname(helpers_path))

print(f"Bundle root: {bundle_root}")

## Read Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Import Volume Path")
dbutils.widgets.text("warehouse_id", "", "Target SQL Warehouse ID")
dbutils.widgets.text("target_parent_path", "", "Target Parent Path")

volume_path = dbutils.widgets.get("volume_path")
warehouse_id = dbutils.widgets.get("warehouse_id")
target_parent_path = dbutils.widgets.get("target_parent_path")

if not volume_path:
    raise ValueError("volume_path parameter is required")
if not warehouse_id:
    raise ValueError("warehouse_id parameter is required")
if not target_parent_path:
    raise ValueError("target_parent_path parameter is required")

print(f"Import volume:      {volume_path}")
print(f"Warehouse ID:       {warehouse_id}")
print(f"Target parent path: {target_parent_path}")

## List Exported Spaces

In [ ]:
exported_dir = os.path.join(volume_path, "exported")

if not os.path.exists(exported_dir):
    raise FileNotFoundError(
        f"Exported directory not found: {exported_dir}\n"
        "Run the transfer notebook first."
    )

json_files = [f for f in os.listdir(exported_dir) if f.endswith(".json")]
print(f"Found {len(json_files)} space(s) to deploy:")
for f in json_files:
    print(f"  - {f}")

## Deploy Spaces

In [ ]:
from databricks.sdk import WorkspaceClient
from helpers.deployer import deploy_all_spaces, write_deployment_manifest

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

print(f"\nDeploying to: {target_parent_path}")
print(f"Using warehouse: {warehouse_id}")

results = deploy_all_spaces(
    client=w,
    import_path=volume_path,
    warehouse_id=warehouse_id,
    parent_path=target_parent_path
)

## Write Deployment Manifest

In [ ]:
manifest_path = write_deployment_manifest(volume_path, results)
print(f"\nDeployment manifest written to: {manifest_path}")

## Display Results

In [ ]:
import pandas as pd

df = pd.DataFrame([r.to_dict() for r in results])
display(df)

## Summary

In [ ]:
success = sum(1 for r in results if r.status == "SUCCESS")
failed = sum(1 for r in results if r.status == "FAILED")
created = sum(1 for r in results if r.action == "CREATED")
updated = sum(1 for r in results if r.action == "UPDATED")

print("=" * 60)
print("DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"Total spaces:  {len(results)}")
print(f"Success:       {success}")
print(f"Failed:        {failed}")
print(f"Created:       {created}")
print(f"Updated:       {updated}")
print(f"\nTarget path:   {target_parent_path}")
print(f"Manifest:      {manifest_path}")

if failed > 0:
    print("\n⚠️  Some deployments failed. Check the results above for details.")

print("\nNext step: Run Tgt_03_Apply_Permissions to set up ACLs.")